# 🧠 Sentiment Analysis: BERTimbau (Gold Standard)
Este notebook gerencia o processo de criação do 'Gold Standard', teste de baseline e fine-tuning do modelo BERTimbau para o chat da Bundesliga.

In [ ]:
import pandas as pd
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

# Verificar GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

## 1. Baseline Test: BERTimbau 'Clean'
Testamos o modelo base (sem treinamento específico) contra nosso Gold Standard.

In [ ]:
gold_df = pd.read_csv('../data/processed/consolidated/gold_standard_labeled.csv')

model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)

def get_prediction(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.argmax(logits, dim=1).item()

print("Executando inferência baseline...")
tqdm.pandas()
gold_df['pred_baseline'] = gold_df['mensagem'].progress_apply(get_prediction)

print("\nRelatório de Classificação (Baseline):")
print(classification_report(gold_df['sentiment_manual'], gold_df['pred_baseline']))

## 2. Fine-tuning (Opcional)
Se o baseline for ruim (F1 < 0.6), realizamos o ajuste fino usando os 500 exemplos.

In [ ]:
# TODO: Implementar script de fine-tuning se necessário